In [18]:
import pandas as pd

X_train = pd.read_csv("../data/training.csv")
X_test = pd.read_csv("../data/test-ans.csv")

y_train = X_train.real
y_test = X_test.real

X_test_id = X_test.id

X_train.drop(columns=['id', 'real'], inplace=True)
X_test.drop(columns=['id', 'real'], inplace=True)

In [8]:
X_test

,text,length
0,"""As an employee, I would never have to say any...",1024
1,Guns N' Roses took the stage in New York City ...,43
2,The Supreme Court of Massachusetts is consider...,930
3,"The first year out, the most important thing h...",1024
4,"""We are going to fight back tomorrow,"" said on...",1024
...,...,...
995,"If Donald Trump comes to Washington, he'll hea...",1024
996,"What would he say, when he sees someone in the...",266
997,"Downtown\n\nHilton, one of the most popular ho...",584
998,In his speech to the U.S. Constitution Congres...,1024


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

def add_feature(X, feature_to_add):
    """
    Returns sparse feature matrix with added feature.
    feature_to_add can also be a list of features.
    """
    from scipy.sparse import csr_matrix, hstack
    return hstack([X, csr_matrix(feature_to_add).T], 'csr')

vect = TfidfVectorizer()
X_train_vect = vect.fit_transform(X_train.text)
X_train_vect = add_feature(X_train_vect, X_train.length)

X_test_vect = vect.transform(X_test.text)
X_test_vect = add_feature(X_test_vect, X_test.length)

In [14]:
X_train_vect

<4000x37872 sparse matrix of type '<class 'numpy.float64'>'
	with 844016 stored elements in Compressed Sparse Row format>

In [43]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate

cv_results = cross_validate(
    DecisionTreeClassifier(), # the model or the pipeline
    #RandomForestClassifier(random_state=0),
    X_train_vect, # the data that will be used in the cross-validation
    y_train, # the target
    scoring=["accuracy","f1"], # the metric that we want to evaluate
    cv=10, # the cross-validation fold
)

#print(cv_results)
print('Average Accuracy: {0} +/- {1}'.format(cv_results['test_accuracy'].mean(), cv_results['test_accuracy'].std()))
print('Average F1: {0} +/- {1}'.format(cv_results['test_f1'].mean(), cv_results['test_f1'].std()))

Average Accuracy: 0.99975 +/- 0.0007499999999999841
Average F1: 0.9998204667863554 +/- 0.0005385996409335681


In [44]:
model = DecisionTreeClassifier()
model.fit(X_train_vect, y_train)

DecisionTreeClassifier()

In [45]:
from sklearn.metrics import accuracy_score, f1_score

y_pred = model.predict(X_test_vect)
print('Accuracy =', accuracy_score(y_test, y_pred))
print('f1 =', f1_score(y_test, y_pred, average='binary'))

df = pd.DataFrame(data={'id': X_test_id, 'Predicted_Result' : y_pred}).set_index('id')
df.to_csv('Final_Results.csv')

Accuracy = 1.0
f1 = 1.0
